# 使用@tool装饰器定义工具

## 1、自定义工具描述：description

举例1：

函数使用@tool装饰器修饰以后，就是一个可以被模型识别的工具了

如下的程序报错了，因为：
在没有提供description参数的情况下，必须提供docstring

In [14]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool
def get_weather(city : str):

    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

ValueError: Function must have a docstring if description not provided.

修改为：

In [13]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool
def get_weather(city : str):
    """获取城市的天气"""
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市的天气',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

举例2：使用description参数

In [4]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool(description="获取具体城市的天气情况")
def get_weather(city : str):
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气情况',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

举例3：
在同时声明了description和docstring的情况下，description的优先级更高(也就是在程序运行结果会优先显示description)

In [5]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool(description="获取具体城市的天气情况")
def get_weather(city : str):
    """获取城市的天气"""
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气情况',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

举例4：

当我们没有向@tool传递description参数时，默认情况下，@tool会将docstring整体视为description，这种情况下需要添加parse_docstring参数，设置为True，docstring就会被解析，填充到相应字段描述中

要注意：不使用@tool装饰器时，docstring的写法不符合google-style的时候会被视为普通文本，作为description，但是如果使用@tool装饰器时，docstring的写法不符合google-style时，程序就会报错

In [18]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool(parse_docstring=True)
def get_weather(city : str):
    """
    获取城市的天气

    Args:
        city : 城市
    """
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市的天气',
        'parameters': {
            'properties': {'city': {'description': '城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

举例5：

如果在@tool中设置了description的情况下，parse_docstring是True的时候，description的优先级仍然是最高的

实际开发中一般在docstring中写了工具描述就不会再去设置description了

In [9]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool(parse_docstring=True,description="获取具体城市的天气")
def get_weather(city : str):
    """
    获取城市的天气

    Args:
        city : 城市
    """
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气',
        'parameters': {
            'properties': {'city': {'description': '城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 2、更改工具名称: name_or_callable

举例：

一般实际开发中，还是使用函数名作为工具名称，但是这里就是说一下可以通过使用name_or_callable去更改函数的名称

需要注意的是：命名的时候不要使用config或runtime作为参数名，这些是LangChain内部保留的

In [11]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool(parse_docstring=True,name_or_callable="getWeather")
def get_weather(city : str):
    """
    获取城市的天气

    Args:
        city : 城市
    """
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '获取城市的天气',
        'parameters': {
            'properties': {'city': {'description': '城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 3、自定义args_schema

### 3.1 使用Pydantic模型定义

举例1：

In [4]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

from pydantic import BaseModel

class WeatherInput(BaseModel):
    city : str

@tool(args_schema=WeatherInput)
def get_weather(city : str):
    """
    获取城市的天气
    """
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市的天气',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

举例2：

In [6]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from pydantic import BaseModel, Field


class WeatherInput(BaseModel):
    city : str = Field(
        description="具体城市的天气",
        default="江西"
    )

@tool(args_schema=WeatherInput)
def get_weather(city : str):
    """
    获取城市的天气
    """
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市的天气',
        'parameters': {
            'properties': {'city': {'default': '江西', 'description': '具体城市的天气', 'type': 'string'}},
            'type': 'object'
        }
    }
}

举例3：

In [13]:
from typing import Literal


class WeatherInput(BaseModel):
    city : str = Field(
        default="江西",
        description="具体的城市"
    )
    unit : Literal["celsius","fahrenheit"]
    include_forecast : bool = Field(
        default=False,
        description="是否包含未来五天的天气预报"
    )

@tool(args_schema=WeatherInput)
def get_weather(city : str,unit : str = "celsius",include_forecast : bool = True):
    """
    获取城市的天气
    """
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市的天气',
        'parameters': {
            'properties': {
                'city': {'default': '江西', 'description': '具体的城市', 'type': 'string'},
                'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'},
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五天的天气预报',
                    'type': 'boolean'
                }
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}

### 3.2 使用Json Schema定义

举例1：

In [12]:
json_schema = {
            'properties': {
                'city': {'default': '江西', 'description': '具体的城市11', 'type': 'string'},
                'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'},
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五天的天气预报',
                    'type': 'boolean'
                }
            },
            'required': ['unit'],
            'type': 'object'
        }


@tool(args_schema=json_schema)
def get_weather(city : str,unit : str = "celsius",include_forecast : bool = True):
    """
    获取城市的天气
    """
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市的天气',
        'parameters': {
            'properties': {
                'city': {'default': '江西', 'description': '具体的城市11', 'type': 'string'},
                'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'},
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五天的天气预报',
                    'type': 'boolean'
                }
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}